# MovieSentiment — DistilBERT Fine-tuning (Colab/Kaggle GPU)

Run this notebook on Colab (T4 free) or Kaggle (T4, 30 h/week).

**Runtime:** ~25 min on T4 for 3 epochs over 25k train reviews.

After training, download `distilbert/` and `metrics/transformer.json` back into the local repo.

In [ ]:
# 1. Clone repo (replace with your fork URL)
!git clone https://github.com/Cryptic2-0/moviesentiment.git
%cd moviesentiment

In [ ]:
# 2. Install deps (transformers + torch already on Colab; add project extras)
!pip install -e '.[dev]' --quiet

In [ ]:
# 3. Pull processed data from DVC remote (S3)
#    Set AWS creds as Colab Secrets (wrench icon) or paste here temporarily.
import os
# os.environ['AWS_ACCESS_KEY_ID'] = '...'
# os.environ['AWS_SECRET_ACCESS_KEY'] = '...'

!dvc pull data/processed/

In [ ]:
# 4. Verify GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 5. Train — MLflow logs to local SQLite (mlflow.db)
from moviesentiment.models.transformer import train_transformer
train_transformer()

In [ ]:
# 6. Zip artifacts for download
!zip -r distilbert_artifacts.zip models/distilbert/ metrics/transformer.json mlflow.db mlruns/

from google.colab import files
files.download('distilbert_artifacts.zip')

## After downloading

Unzip into the local repo root, then:

```powershell
# Push model artifacts to DVC remote
.venv\Scripts\dvc.exe add models/distilbert
.venv\Scripts\dvc.exe push

# Commit the .dvc pointer file
git add models/distilbert.dvc metrics/transformer.json dvc.lock
git commit -m 'feat(day6): distilbert fine-tuning artifacts'
git push
```